# Self-Attention from Scratch (Implementation Example)

**Problem:** compute the self-attention output for a toy "sentence" of four words, step by step, with NumPy. This is the code shown on the *Implementation Example* slide of the lecture.

**How to run:** this notebook runs in your browser on the course website. Use **Run → Run All Cells**, or press **Shift+Enter** to run one cell at a time.

**Expected result:** a 4 x 3 array, one attention output vector per word, with first row `[0.98522025 1.74174051 0.75652026]` (the same numbers as on the slide).

## 1. Imports
We need NumPy for arrays and random numbers, and SciPy's `softmax` to turn scores into weights that add up to 1.

In [1]:
from numpy import array
from numpy import random
from numpy import dot
from scipy.special import softmax

## 2. Word embeddings
Each word is represented by a small vector (its *encoder representation*). We stack the four vectors into one array: one row per word.

In [2]:
# encoder representations of four different words
word_1 = array([1, 0, 0])
word_2 = array([0, 1, 0])
word_3 = array([1, 1, 0])
word_4 = array([0, 0, 1])

# stacking the word embeddings into a single array
words = array([word_1, word_2, word_3, word_4])

## 3. Queries, keys and values
In a trained transformer the weight matrices $W_Q$, $W_K$, $W_V$ are learnt. Here we fill them with random integers (with a fixed seed, so the result is always the same) and multiply the word vectors by them to get the queries $Q$, keys $K$ and values $V$.

In [3]:
# generating the weight matrices
random.seed(42)
W_Q = random.randint(3, size=(3, 3))
W_K = random.randint(3, size=(3, 3))
W_V = random.randint(3, size=(3, 3))

# generating the queries, keys and values
Q = words @ W_Q
K = words @ W_K
V = words @ W_V

## 4. Scaled dot-product attention
Each query is compared with every key (dot product). The scores are divided by $\sqrt{d_k}$ (here $d_k = 3$, the length of a key) and turned into weights with softmax. The output for each word is the weighted sum of the value vectors:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [4]:
# scoring the query vectors against all key vectors
scores = Q @ K.transpose()

# computing the weights by a softmax operation (scaled by the square root of the key length)
weights = softmax(scores / K.shape[1] ** 0.5, axis=1)

# computing the attention by a weighted sum of the value vectors
attention = weights @ V

print(attention)

[[0.98522025 1.74174051 0.75652026]
 [0.90965265 1.40965265 0.5       ]
 [0.99851226 1.75849334 0.75998108]
 [0.99560386 1.90407309 0.90846923]]


### What do you notice?
Look at the second row. Why are its numbers different from the other rows?

<details>
<summary>Answer</summary>

Each row is a weighted average of the rows of `V`, and the weights show how strongly that word's query matches each key. Words 1, 3 and 4 put most of their weight (75–91%) on word 3, so their outputs are close to word 3's value vector `[1, 2, 1]`. Word 2's query matches the keys of words 1 and 3 equally (weights of about 0.455 each), so its output `[0.90965265 1.40965265 0.5]` is closer to an even mix of their value vectors. You can print `weights` to check. Try changing `random.seed(42)` to another number: the weights, and therefore the outputs, change.
</details>